In [1]:
import h5py
import traceback
import numpy as np
import plotly.graph_objects as go
import matplotlib as mpl
mpl.use('Qt5Agg')  # Use Qt5 backend for GUI to work properly
import matplotlib.pyplot as plt
import scipy.signal as sc
from scipy.optimize import curve_fit
from scipy.ndimage import filters 
import tifffile as tiff
from scipy.signal import find_peaks
import plotly.express as px
import os
import csv
import pandas as pd
from roipoly import MultiRoi
import plotly.io as pio
from plotly.subplots import make_subplots
from scipy.interpolate import interp1d
import ast
from AnalasysFunction import BurstC,clean_and_parse,motorSp,plotFR,devideTr,plotVolCal,VolToCalIdx,CalSmooth,CorrWindow,ChooseSpk,CalInt,VolToCalIdx,CalAmp,calculate_firing_rate,ChooseCom,LongLIST,SingleSpk,linear_model,quadratic_model,exponential_model,MeanRes,lagOptimaizre
from spike_detection_Qixixn2 import complex_bursts_detection,refine_single_spikes,spike_height_calculation,detect_complex_spikes,refine_all_spikes,plot_trace_with_spikes_pdf,plot_trace_with_spikes_export,plot_trace_with_spikes_html
# Create dictionary
from TRY import LongLIST
from NewinternueronsAnalsys import analyze_block
from scipy.optimize import curve_fit
from sklearn.metrics import mean_squared_error, r2_score
from scipy.stats import pearsonr, linregress,ttest_ind
import pickle
from matplotlib.widgets import Slider, Button

In [2]:
def plot_ca_vol_with_transients_and_spikes(
    ca, vol,
    fs_ca=30, fs_vol=500,
    ca_t=None, vol_t=None,
    spike_idx_v=None,
    transients=None,
    title="Calcium + Voltage (transients highlighted, spikes marked)"
):
    """
    Plot calcium and voltage with secondary y-axis.
    - Voltage on primary y-axis (left)
    - Calcium on secondary y-axis (right)
    - Purple overlays mark calcium transient events (start to end)
    - Black dots mark spike times on voltage trace
    
    Parameters
    ----------
    ca : array-like
        Calcium trace
    vol : array-like
        Voltage trace
    fs_ca : float
        Calcium sampling rate (Hz)
    fs_vol : float
        Voltage sampling rate (Hz)
    ca_t : array-like, optional
        Time axis for calcium (if None, computed from fs_ca)
    vol_t : array-like, optional
        Time axis for voltage (if None, computed from fs_vol)
    spike_idx_v : array-like, optional
        Spike indices in voltage samples
    transients : list of dicts, optional
        List of transient dicts with 'start_idx' and 'end_idx' keys
    title : str
        Figure title
    
    Returns
    -------
    fig : plotly figure
    """
    ca = np.asarray(ca).flatten()
    vol = np.asarray(vol).flatten()

    if ca_t is None:
        ca_t = np.arange(ca.size) / fs_ca
    if vol_t is None:
        vol_t = np.arange(vol.size) / fs_vol

    fig = make_subplots(specs=[[{"secondary_y": True}]])
    
    # ===== ADD VOLTAGE TRACE (PRIMARY Y-AXIS) =====
    fig.add_trace(
        go.Scatter(
            x=vol_t, 
            y=vol, 
            name="Voltage", 
            line=dict(width=1, color="blue"),
            hovertemplate='<b>Voltage</b><br>Time: %{x:.3f}s<br>Value: %{y:.4f}<extra></extra>'
        ),
        secondary_y=False
    )
    
    # ===== ADD SPIKE MARKERS (BLACK DOTS) ON VOLTAGE =====
    if spike_idx_v is not None:
        spike_idx_v = np.asarray(spike_idx_v, dtype=int)
        # Filter valid spike indices
        valid_spikes = spike_idx_v[(spike_idx_v >= 0) & (spike_idx_v < len(vol_t))]
        
        if len(valid_spikes) > 0:
            spike_times = vol_t[valid_spikes]
            spike_values = vol[valid_spikes]
            
            fig.add_trace(
                go.Scatter(
                    x=spike_times,
                    y=spike_values,
                    mode="markers",
                    name=f"Spikes ({len(valid_spikes)})",
                    marker=dict(
                        size=8,
                        color="black",
                        symbol="circle",
                        line=dict(color="white", width=1)
                    ),
                    hovertemplate='<b>Spike</b><br>Time: %{x:.3f}s<br>Voltage: %{y:.4f}<extra></extra>',
                ),
                secondary_y=False
            )
    
    # ===== ADD CALCIUM TRACE (SECONDARY Y-AXIS) =====
    fig.add_trace(
        go.Scatter(
            x=ca_t, 
            y=ca, 
            name="Calcium", 
            line=dict(width=2, color="black"),
            hovertemplate='<b>Calcium</b><br>Time: %{x:.3f}s<br>Value: %{y:.4f}<extra></extra>'
        ),
        secondary_y=True
    )

    # ===== ADD PURPLE OVERLAYS FOR EACH TRANSIENT SEGMENT =====
    if transients is not None:
        for j, tr in enumerate(transients, start=1):
            s, e = tr["start_idx"], tr["end_idx"]
            
            # Ensure indices are within bounds
            s = max(0, s)
            e = min(len(ca_t) - 1, e)
            
            fig.add_trace(
                go.Scatter(
                    x=ca_t[s:e+1], 
                    y=ca[s:e+1],
                    mode="lines",
                    name=f"Ca transient {j}",
                    line=dict(width=4, color="purple"),
                    showlegend=(j == 1),  # keep legend clean
                    hovertemplate='<b>Transient</b><br>Time: %{x:.3f}s<br>Ca: %{y:.4f}<extra></extra>',
                ),
                secondary_y=True
            )

    # ===== UPDATE LAYOUT =====
    fig.update_layout(
        title=title, 
        xaxis_title="Time (s)",
        template="plotly_white",
        hovermode='x unified',
        width=1200,
        height=500,
        font=dict(size=11),
    )
    
    fig.update_yaxes(title_text="<b>Voltage (mV)</b>", secondary_y=False)
    fig.update_yaxes(title_text="<b>Calcium (ΔF/F)</b>", secondary_y=True)
    
    return fig

In [3]:


def gaussian_smooth_1d(x, sigma_samples):
    x = np.asarray(x, float)
    if sigma_samples is None or sigma_samples <= 0:
        return x.copy()
    radius = int(np.ceil(5 * sigma_samples))
    t = np.arange(-radius, radius + 1)
    k = np.exp(-(t**2) / (2 * sigma_samples**2))
    k /= k.sum()
    return np.convolve(x, k, mode="same")

def baseline_mode_hist(x, n_bins=200):
    x = np.asarray(x, float)
    x = x[~np.isnan(x)]
    hist, edges = np.histogram(x, bins=n_bins)
    idx = int(np.argmax(hist))
    return 0.5 * (edges[idx] + edges[idx + 1])

def noise_sd_baseline_mad(x, baseline):
    # robust noise estimate around baseline
    x = np.asarray(x, float)
    r = x - baseline
    mad = np.median(np.abs(r - np.median(r)))
    return 1.4826 * mad if mad > 0 else np.std(r)

def find_calcium_transients(
    ca, fs_ca,
    smooth_sigma_s=0.25,
    k=3.0,
    min_dur_s=0.05,
    merge_gap_s=0.03,
    baseline_bins=200,
    fwhm_search_radius_s=1.0,   # how far to search for half-max crossings
):
    """
    Returns:
      transients: list of dicts with keys:
        start_idx, end_idx (inclusive), peak_idx, baseline, thr,
        peak_val, peak_amp, half_level,
        fwhm_left_idx, fwhm_right_idx, fwhm_samples, fwhm_s,
        hwhm_samples, hwhm_s
    """
    ca = np.asarray(ca, float)
    sigma_samples = smooth_sigma_s * fs_ca
    ca_s = gaussian_smooth_1d(ca, sigma_samples)

    baseline = baseline_mode_hist(ca_s, n_bins=baseline_bins)
    noise_sd = noise_sd_baseline_mad(ca_s, baseline)
    thr = baseline + k * noise_sd

    above = ca_s > thr
    idx = np.flatnonzero(above)
    if idx.size == 0:
        return [], ca_s, baseline, thr

    # contiguous segments
    cuts = np.where(np.diff(idx) > 1)[0]
    starts = np.r_[idx[0], idx[cuts + 1]]
    ends   = np.r_[idx[cuts], idx[-1]]

    # merge segments separated by short gaps
    max_gap = int(round(merge_gap_s * fs_ca))
    merged_s, merged_e = [int(starts[0])], [int(ends[0])]
    for s, e in zip(starts[1:], ends[1:]):
        s, e = int(s), int(e)
        if s - merged_e[-1] - 1 <= max_gap:
            merged_e[-1] = e
        else:
            merged_s.append(s)
            merged_e.append(e)

    # enforce min duration
    min_len = int(round(min_dur_s * fs_ca))
    transients = []

    # max search range for half-max crossings (prevents wandering far away)
    rad = int(round(fwhm_search_radius_s * fs_ca))

    for s, e in zip(merged_s, merged_e):
        if (e - s + 1) < min_len:
            continue

        seg = ca_s[s:e+1]
        peak_idx = s + int(np.nanargmax(seg))

        peak_val = float(ca_s[peak_idx])
        peak_amp = float(peak_val - baseline)

        # Half-max level in the same units as ca_s
        half_level = float(baseline + 0.5 * peak_amp)

        # ----- Find half-max crossings (FWHM) -----
        # Left crossing: search from peak back toward s (or limited by radius)
        left_bound = max(s, peak_idx - rad)
        left_idx = peak_idx
        for j in range(peak_idx, left_bound - 1, -1):
            if ca_s[j] <= half_level:
                left_idx = j
                break

        # Right crossing: search from peak forward toward e (or limited by radius)
        right_bound = min(e, peak_idx + rad)
        right_idx = peak_idx
        for j in range(peak_idx, right_bound + 1):
            if ca_s[j] <= half_level:
                right_idx = j
                break

        fwhm_samples = int(max(0, right_idx - left_idx))
        fwhm_s = float(fwhm_samples / fs_ca)

        # HWHM is half of FWHM
        hwhm_samples = float(fwhm_samples / 2.0)
        hwhm_s = float(fwhm_s / 2.0)

        transients.append(dict(
            start_idx=int(s),
            end_idx=int(e),
            peak_idx=int(peak_idx),
            baseline=float(baseline),
            thr=float(thr),

            peak_val=peak_val,
            peak_amp=peak_amp,
            half_level=half_level,

            fwhm_left_idx=int(left_idx),
            fwhm_right_idx=int(right_idx),
            fwhm_samples=int(fwhm_samples),
            fwhm_s=fwhm_s,

            hwhm_samples=hwhm_samples,
            hwhm_s=hwhm_s
        ))

    return transients, ca_s, baseline, thr


def fr_corr_and_windowed(
    cal_trace, fs_ca,
    spike_idx_v, fs_v,
    window_dur_s=10.0,
    fr_bin_s=0.2,
    fr_smooth_sigma_s=0.3,
    ca_smooth_sigma_s=0.0,
):
    """
    Divide cal_trace and spike_idx_v into non-overlapping windows (default 10 sec),
    compute Pearson r for each window independently.

    Returns dict with:
      window_r_list, window_p_list, window_times (centers),
      n_windows, params
    """
    cal_trace = np.asarray(cal_trace, float)
    n_ca = cal_trace.size
    t_ca = np.arange(n_ca) / float(fs_ca)
    duration_s = t_ca[-1]

    # Define non-overlapping windows
    window_samples = int(np.round(window_dur_s * fs_ca))
    n_windows = int(np.floor(n_ca / window_samples))

    if n_windows == 0:
        return {
            "window_r_list": np.array([], float),
            "window_p_list": np.array([], float),
            "window_times": np.array([], float),
            "n_windows": 0,
            "params": dict(window_dur_s=window_dur_s, fs_ca=fs_ca, fs_v=fs_v)
        }

    window_r_list = []
    window_p_list = []
    window_times = []

    spike_idx_v = np.asarray(spike_idx_v, dtype=int)

    for w in range(n_windows):
        start_ca = w * window_samples
        end_ca = start_ca + window_samples
        
        # Clamp to valid range
        end_ca = min(end_ca, n_ca)

        # Extract window
        cal_win = cal_trace[start_ca:end_ca]
        if cal_win.size == 0:
            continue

        # Smooth Ca (optional)
        if ca_smooth_sigma_s and ca_smooth_sigma_s > 0:
            ca_win = gaussian_smooth(cal_win, ca_smooth_sigma_s * fs_ca)
        else:
            ca_win = cal_win.copy()

        # Time axis for this window
        t_win = np.arange(start_ca, end_ca) / float(fs_ca)
        n_win = len(t_win)

        # Convert window time to voltage indices
        t_start_s = t_win[0]
        t_end_s = t_win[-1]
        idx_start_v = int(t_start_s * fs_v)
        idx_end_v = int(t_end_s * fs_v)

        # Filter spikes in this window
        spikes_in_win = spike_idx_v[
            (spike_idx_v >= idx_start_v) & (spike_idx_v < idx_end_v)
        ]

        # Compute FR on Ca timebase for this window
        if spikes_in_win.size == 0:
            fr_win = np.zeros(n_win, float)
        else:
            # Spike times in seconds
            spike_t = spikes_in_win / float(fs_v)

            # Bin spikes
            bin_w = max(float(fr_bin_s), 1e-6)
            edges = np.arange(t_start_s, t_end_s + bin_w, bin_w)
            if edges.size < 2:
                edges = np.array([t_start_s, t_end_s + bin_w])

            counts, _ = np.histogram(spike_t, bins=edges)
            fr = counts / bin_w
            t_fr = (edges[:-1] + edges[1:]) / 2.0

            # Smooth FR
            if fr_smooth_sigma_s and fr_smooth_sigma_s > 0:
                fr = gaussian_smooth(fr, sigma_samples=(fr_smooth_sigma_s / bin_w))

            # Interpolate to Ca axis
            fr_win = np.interp(t_win, t_fr, fr, left=fr[0], right=fr[-1])

        # Compute Pearson for this window
        m = np.isfinite(fr_win) & np.isfinite(ca_win)
        if m.sum() < 5:
            r, p = np.nan, np.nan
        else:
            x = fr_win[m]
            y = ca_win[m]
            if np.std(x) == 0 or np.std(y) == 0:
                r, p = np.nan, np.nan
            else:
                r, p = pearsonr(x, y)

        window_r_list.append(float(r) if np.isfinite(r) else np.nan)
        window_p_list.append(float(p) if np.isfinite(p) else np.nan)
        window_times.append((t_start_s + t_end_s) / 2.0)  # center of window

    return {
        "window_r_list": np.asarray(window_r_list, float),
        "window_p_list": np.asarray(window_p_list, float),
        "window_times": np.asarray(window_times, float),
        "n_windows": len(window_r_list),
        "params": dict(
            window_dur_s=window_dur_s,
            fr_bin_s=fr_bin_s,
            fr_smooth_sigma_s=fr_smooth_sigma_s,
            ca_smooth_sigma_s=ca_smooth_sigma_s,
            fs_ca=fs_ca,
            fs_v=fs_v,
        )
    }

In [4]:
def analyze_cell_calcium_transients(
    cell_path,
    trace_vol,
    trace_cal,
    spike_indices,
    frame_rate_vol=500,
    frame_rate_cal=30,
    window_ms=100,
    smooth_sigma_s=0,
    k=2.0,
    min_dur_s=0.25,
    merge_gap_s=0.15,
    baseline_bins=200,
    name=''
):
    """
    Detect calcium transients and extract aligned voltage traces, spikes, and AUC.
    
    Parameters
    ----------
    cell_path : str
        Path to cell directory (for saving results)
    trace_vol : np.ndarray
        Voltage trace
    trace_cal : np.ndarray
        Calcium trace
    spike_indices : np.ndarray
        Spike times in voltage samples
    frame_rate_vol : int
        Voltage sampling rate (Hz)
    frame_rate_cal : int
        Calcium sampling rate (Hz)
    window_ms : int
        Window around transient peak (±ms)
    smooth_sigma_s : float
        Smoothing sigma in seconds for Ca detection
    k : float
        Threshold multiplier (k * noise_sd above baseline)
    min_dur_s : float
        Minimum transient duration in seconds
    merge_gap_s : float
        Merge transients separated by this gap
    baseline_bins : int
        Number of bins for mode-based baseline estimation
    
    Returns
    -------
    results_dict : dict with keys:
        - 'transients': list of transient dicts
        - 'transient_cal_traces': list of Ca traces ±window_ms around peak
        - 'transient_vol_traces': list of V traces ±window_ms around peak
        - 'transient_spikes': list of spike indices within each transient
        - 'transient_auc': list of AUC values for each transient
        - 'transient_auc_above_baseline': list of AUC above baseline for each transient
        - 'transient_peak_amplitude': list of peak amplitude above baseline
        - 'transient_duration_s': list of duration in seconds for each transient
        - 'ca_smooth': smoothed Ca trace used for detection
        - 'detection_params': dict of all parameters used
    """
    
    trace_vol = np.asarray(trace_vol, dtype=float)
    trace_cal = np.asarray(trace_cal, dtype=float)
    spike_indices = np.asarray(spike_indices, dtype=int)
    
    # print(f"\n{'='*70}")
    # print(f"CALCIUM TRANSIENT ANALYSIS")
    # print(f"{'='*70}")
    # print(f"Voltage trace: {len(trace_vol)} samples ({len(trace_vol)/frame_rate_vol:.2f} s)")
    # print(f"Calcium trace: {len(trace_cal)} samples ({len(trace_cal)/frame_rate_cal:.2f} s)")
    # print(f"Spikes detected: {len(spike_indices)}")
    
    # ===== 1. DETECT CALCIUM TRANSIENTS =====
    transients, ca_smooth, baseline, thr = find_calcium_transients(
        trace_cal,
        fs_ca=frame_rate_cal,
        smooth_sigma_s=smooth_sigma_s,
        k=k,
        min_dur_s=min_dur_s,
        merge_gap_s=merge_gap_s,
        baseline_bins=baseline_bins
    )
    
    # print(f"\n✓ Transients detected: {len(transients)}")
    # print(f"  Baseline: {baseline:.4f}")
    # print(f"  Threshold: {thr:.4f}")
    
    # ===== 2. EXTRACT ALIGNED TRACES AND CALCULATE AUC =====
    transient_cal_traces = []
    transient_vol_traces = []
    transient_spikes_list = []
    transient_auc_list = []
    transient_auc_baseline_list = []
    transient_peak_amp_list = []
    transient_fwhm_list = []
    transient_dur_s_list = []
    
    # Convert window from ms to samples
    window_ca_samples = int(np.round(window_ms * frame_rate_cal / 1000))
    window_vol_samples = int(np.round(window_ms * frame_rate_vol / 1000))
    
    # print(f"\nExtracting ±{window_ms} ms windows around each transient peak:")
    # print(f"  Ca window: ±{window_ca_samples} samples")
    # print(f"  Vol window: ±{window_vol_samples} samples")
    
    # Create time axis for alignment
    t_vol = np.arange(len(trace_vol)) / frame_rate_vol
    t_cal = np.arange(len(trace_cal)) / frame_rate_cal
    
    for trans_idx, transient in enumerate(transients):
        peak_idx_ca = transient['peak_idx']
        start_idx_ca = transient['start_idx']
        end_idx_ca = transient['end_idx']
        
        # Extract Ca window (±window_ms around peak)
        ca_start = max(0, start_idx_ca - window_ca_samples)
        ca_end = min(len(trace_cal), end_idx_ca + window_ca_samples + 1)
        ca_window = trace_cal[ca_start:ca_end]
        
        # Convert peak time to voltage indices
        peak_time_s = t_cal[peak_idx_ca]
        peak_idx_vol = np.argmin(np.abs(t_vol - peak_time_s))
        Start_time_s = t_cal[start_idx_ca]
        Start_idx_vol = np.argmin(np.abs(t_vol - Start_time_s))
        End_time_s = t_cal[end_idx_ca]
        End_idx_vol = np.argmin(np.abs(t_vol - End_time_s))
        
        # Extract Vol window (±window_ms around corresponding time)
        vol_start = max(0, Start_idx_vol - window_vol_samples)
        vol_end = min(len(trace_vol), End_idx_vol + window_vol_samples + 1)
        vol_window = trace_vol[vol_start:vol_end]
        
        # Find spikes within this transient (using original indices)
        spikes_in_transient = spike_indices[
            (spike_indices >= vol_start) & (spike_indices < vol_end)
        ]
        # Adjust spike indices relative to window start
        spikes_relative = spikes_in_transient - vol_start
        
        # ===== CALCULATE AUC =====
        # AUC of full calcium window
       
        t_ca_transient = np.arange(len(trace_cal[start_idx_ca:end_idx_ca])) / frame_rate_cal
        auc_full = np.trapz(trace_cal[start_idx_ca:end_idx_ca], t_ca_transient)
        
        # AUC above baseline (use same length as ca_window)
        t_ca_window = np.arange(len(ca_window)) / frame_rate_cal
        ca_above_baseline = np.maximum(ca_window - baseline, 0)
        auc_above_baseline = np.trapz(ca_above_baseline, t_ca_window)
        
        # Peak amplitude above baseline
        peak_amp = np.max(ca_window) - baseline
        
        # Duration of transient
        trans_dur = (end_idx_ca - start_idx_ca + 1) / frame_rate_cal
        fwhm_s = transient['fwhm_s']
        # Store results
        transient_cal_traces.append(ca_window)
        transient_vol_traces.append(vol_window)
        transient_spikes_list.append(spikes_relative)
        transient_auc_list.append(auc_full)
        transient_auc_baseline_list.append(auc_above_baseline)
        transient_peak_amp_list.append(peak_amp)
        transient_dur_s_list.append(trans_dur)
        transient_fwhm_list.append(fwhm_s)
        
        if (trans_idx + 1) % 10 == 0 or trans_idx == len(transients) - 1:
            print(f"  Transient {trans_idx + 1}/{len(transients)}: "
                  f"AUC={auc_full:.2f} | AUC(↑baseline)={auc_above_baseline:.2f} | "
                  f"Peak={peak_amp:.4f} | Duration={trans_dur:.3f}s | "
                  f"Spikes={len(spikes_relative)}")
    
    # ===== 3. COMPILE RESULTS =====
    results_dict = {
        'transients': transients,
        'transient_cal_traces': transient_cal_traces,
        'transient_vol_traces': transient_vol_traces,
        'transient_spikes': transient_spikes_list,
        'transient_auc': np.array(transient_auc_list, dtype=float),
        'transient_auc_above_baseline': np.array(transient_auc_baseline_list, dtype=float),
        'transient_peak_amplitude': np.array(transient_peak_amp_list, dtype=float),
        'transient_duration_s': np.array(transient_dur_s_list, dtype=float),
        'transient_fwhm_s': np.array(transient_fwhm_list, dtype=float),
        'ca_smooth': ca_smooth,
        'baseline': baseline,
        'threshold': thr,
        'detection_params': {
            'window_ms': window_ms,
            'smooth_sigma_s': smooth_sigma_s,
            'k': k,
            'min_dur_s': min_dur_s,
            'merge_gap_s': merge_gap_s,
            'baseline_bins': baseline_bins,
            'frame_rate_vol': frame_rate_vol,
            'frame_rate_cal': frame_rate_cal,
        }
    }
    
    # ===== 4. SAVE RESULTS =====
    save_path = os.path.join(cell_path, f'calcium_transient_analysis{name}.pkl')
    with open(save_path, 'wb') as f:
        pickle.dump(results_dict, f)
    
    # print(f"\n✓ Results saved to: {save_path}")
    # print(f"\nSummary Statistics:")
    # print(f"  Total transients extracted: {len(transient_cal_traces)}")
    # print(f"  Transients with spikes: {sum(1 for s in transient_spikes_list if len(s) > 0)}")
    # print(f"  Total spikes in transients: {sum(len(s) for s in transient_spikes_list)}")
    # print(f"  Mean AUC: {np.nanmean(transient_auc_list):.2f} ± {np.nanstd(transient_auc_list):.2f}")
    # print(f"  Mean AUC (above baseline): {np.nanmean(transient_auc_baseline_list):.2f} ± {np.nanstd(transient_auc_baseline_list):.2f}")
    # print(f"  Mean peak amplitude: {np.nanmean(transient_peak_amp_list):.4f} ± {np.nanstd(transient_peak_amp_list):.4f}")
    # print(f"  Mean duration: {np.nanmean(transient_dur_s_list):.3f} ± {np.nanstd(transient_dur_s_list):.3f} s")
    
    return results_dict


def plot_transient_summary_with_auc(results_dict, num_to_plot=20, figsize=(18, 4)):
    """
    Plot first N transients with Ca (AUC info), Vol traces and spike markers.
    """
    transients = results_dict['transients']
    cal_traces = results_dict['transient_cal_traces']
    vol_traces = results_dict['transient_vol_traces']
    spike_lists = results_dict['transient_spikes']
    auc_vals = results_dict['transient_auc']
    auc_baseline = results_dict['transient_auc_above_baseline']
    peak_amps = results_dict['transient_peak_amplitude']
    frame_rate_vol = results_dict['detection_params']['frame_rate_vol']
    frame_rate_cal = results_dict['detection_params']['frame_rate_cal']
    baseline = results_dict['baseline']
    
    num_to_plot = min(num_to_plot, len(transients))
    n_cols = 4
    n_rows = int(np.ceil(num_to_plot / n_cols))
    
    fig, axes = plt.subplots(n_rows, n_cols * 2, figsize=(figsize[0], figsize[1] * n_rows))
    axes = axes.flatten()
    
    for i in range(num_to_plot):
        # Left: Calcium trace
        ax_ca = axes[i * 2]
        ca_trace = cal_traces[i]
        t_ca = np.arange(len(ca_trace)) / frame_rate_cal * 1000  # ms
        ax_ca.fill_between(t_ca, baseline, ca_trace, alpha=0.3, color='green', label='Transient')
        ax_ca.plot(t_ca, ca_trace, 'g-', lw=2, label='Ca')
        ax_ca.axhline(baseline, color='k', linestyle='--', alpha=0.5, label='Baseline')
        ax_ca.set_title(
            f'Transient {i+1} (Ca)\n'
            f'AUC={auc_vals[i]:.2f} | AUC(↑bsl)={auc_baseline[i]:.2f}\n'
            f'Peak={peak_amps[i]:.4f}'
        )
        ax_ca.set_xlabel('Time (ms)')
        ax_ca.set_ylabel('Calcium')
        ax_ca.legend(fontsize=8, loc='upper right')
        ax_ca.grid(True, alpha=0.3)
        
        # Right: Voltage trace with spikes
        ax_vol = axes[i * 2 + 1]
        vol_trace = vol_traces[i]
        spike_idx = spike_lists[i]
        
        t_vol = np.arange(len(vol_trace)) / frame_rate_vol * 1000  # ms
        ax_vol.plot(t_vol, vol_trace, 'b-', lw=1.5, label='Voltage')
        
        # Mark spikes
        if len(spike_idx) > 0:
            ax_vol.scatter(t_vol[spike_idx], vol_trace[spike_idx], 
                          c='r', s=100, marker='x', label=f'Spikes ({len(spike_idx)})', 
                          zorder=5, linewidths=2)
        
        ax_vol.set_title(f'Transient {i+1} (Voltage)')
        ax_vol.set_xlabel('Time (ms)')
        ax_vol.set_ylabel('Voltage')
        ax_vol.legend(fontsize=8)
        ax_vol.grid(True, alpha=0.3)
    
    # Hide unused subplots
    for j in range(num_to_plot * 2, len(axes)):
        axes[j].set_visible(False)
    
    plt.tight_layout()
    return fig


def plot_auc_statistics(results_dict):
    """
    Plot distribution of AUC, peak amplitude, and relationship to spike count.
    """
    auc_vals = results_dict['transient_auc']
    auc_baseline = results_dict['transient_auc_above_baseline']
    peak_amps = results_dict['transient_peak_amplitude']
    spike_counts = np.array([len(s) for s in results_dict['transient_spikes']])
    durations = results_dict['transient_duration_s']
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    
    # 1. AUC distribution
    axes[0, 0].hist(auc_vals, bins=20, color='blue', alpha=0.7, edgecolor='black')
    axes[0, 0].set_xlabel('Full AUC')
    axes[0, 0].set_ylabel('Count')
    axes[0, 0].set_title('Distribution of Full AUC')
    axes[0, 0].grid(True, alpha=0.3)
    
    # 2. AUC above baseline distribution
    axes[0, 1].hist(auc_baseline, bins=20, color='green', alpha=0.7, edgecolor='black')
    axes[0, 1].set_xlabel('AUC (above baseline)')
    axes[0, 1].set_ylabel('Count')
    axes[0, 1].set_title('Distribution of AUC Above Baseline')
    axes[0, 1].grid(True, alpha=0.3)
    
    # 3. Peak amplitude distribution
    axes[0, 2].hist(peak_amps, bins=20, color='orange', alpha=0.7, edgecolor='black')
    axes[0, 2].set_xlabel('Peak Amplitude (ΔF)')
    axes[0, 2].set_ylabel('Count')
    axes[0, 2].set_title('Distribution of Peak Amplitude')
    axes[0, 2].grid(True, alpha=0.3)
    
    # 4. AUC vs Spike Count
    axes[1, 0].scatter(spike_counts, auc_vals, alpha=0.5, s=30)
    axes[1, 0].set_xlabel('Spike Count')
    axes[1, 0].set_ylabel('Full AUC')
    axes[1, 0].set_title('AUC vs Spike Count')
    axes[1, 0].grid(True, alpha=0.3)
    
    # 5. AUC (above baseline) vs Spike Count
    axes[1, 1].scatter(spike_counts, auc_baseline, alpha=0.5, s=30, color='green')
    axes[1, 1].set_xlabel('Spike Count')
    axes[1, 1].set_ylabel('AUC (above baseline)')
    axes[1, 1].set_title('AUC Above Baseline vs Spike Count')
    axes[1, 1].grid(True, alpha=0.3)
    
    # 6. Peak amplitude vs Duration
    axes[1, 2].scatter(durations, peak_amps, alpha=0.5, s=30, color='orange')
    axes[1, 2].set_xlabel('Duration (s)')
    axes[1, 2].set_ylabel('Peak Amplitude')
    axes[1, 2].set_title('Peak Amplitude vs Duration')
    axes[1, 2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    return fig

In [63]:
def find_calcium_transients_hp(
    ca, fs_ca,
    k=2.0,
    min_dur_s=0.1,
    merge_gap_s=0.03,
    baseline_bins=200,
    median_filter_size_ms=20,
    smooth_sigma_s=0.07
):
    """
    Pipeline:
    1. Median filter → find peaks and segment events
    2. Gaussian smooth original trace → refine start/end points
    
    Returns:
      transients: list of dicts with keys:
        start_idx, end_idx, peak_idx, baseline, thr, peak_amp, fwhm_samples, fwhm_s
    """
    from scipy.ndimage import median_filter
    
    ca = np.asarray(ca, float)
    
    # Step 1: MEDIAN FILTER (for peak detection)
    if median_filter_size_ms and median_filter_size_ms > 0:
        median_samples = int(np.round(median_filter_size_ms * fs_ca / 1000))
        ca_filtered = median_filter(ca, size=median_samples)
    else:
        ca_filtered = ca.copy()
    
    # Step 2: GAUSSIAN SMOOTH (for boundary refinement)
    sigma_samples = smooth_sigma_s * fs_ca
    ca_smooth = gaussian_smooth_1d(ca, sigma_samples)
    
    # Step 3: FIND BASELINE & THRESHOLD (use filtered signal)
    baseline = baseline_mode_hist(ca_filtered, n_bins=baseline_bins)
    noise_sd = noise_sd_baseline_mad(ca_filtered, baseline)
    thr = baseline + k * noise_sd
    
    # Step 4: DETECT PEAKS on filtered signal
    above_thr = ca_filtered > thr
    idx = np.flatnonzero(above_thr)
    
    if idx.size == 0:
        return [], ca_smooth, baseline, thr
    
    # Step 5: GROUP into contiguous segments
    cuts = np.where(np.diff(idx) > 1)[0]
    starts = np.r_[idx[0], idx[cuts + 1]]
    ends   = np.r_[idx[cuts], idx[-1]]
    
    #Step 6: MERGE nearby segments
    max_gap = int(round(merge_gap_s * fs_ca))
    merged_s, merged_e = [int(starts[0])], [int(ends[0])]
    for s, e in zip(starts[1:], ends[1:]):
        s, e = int(s), int(e)
        if s - merged_e[-1] - 1 <= max_gap:
            merged_e[-1] = e
        else:
            merged_s.append(s)
            merged_e.append(e)
    
    #Step 7: REFINE start/end on SMOOTHED original trace & calculate FWHM
    min_len = int(round(min_dur_s * fs_ca))
    transients = []
    
    for s, e in zip(merged_s, merged_e):
        if (e - s + 1) < min_len:
            continue
        
        # Find peak index in filtered signal
        seg = ca_filtered[s:e+1]
        peak_idx = s + int(np.nanargmax(seg))
        peak_amp = ca_smooth[peak_idx] - baseline
        
        # ===== REFINE START: search backward on smoothed trace =====
        refined_s = s
        search_radius = int(0.75 * fs_ca)
        for j in range(s, max(0, s - search_radius), -1):
            if ca_smooth[j] < baseline + 0.5 * (thr - baseline):
                refined_s = j
                break
        else:
            refined_s = max(0, s - search_radius)
        
        # ===== REFINE END: search forward on smoothed trace =====
        refined_e = e
        for j in range(e, min(len(ca_smooth), e + search_radius)):
            if ca_smooth[j] < baseline + 0.5 * (thr - baseline):
                refined_e = j
                break
        else:
            refined_e = min(len(ca_smooth) - 1, e + search_radius)
        
        refined_s = max(0, refined_s)
        refined_e = min(len(ca_smooth) - 1, refined_e)
        
        if refined_e - refined_s + 1 >= min_len:
            # ===== CALCULATE FWHM =====
            peak_val = ca_smooth[peak_idx]
            half_max = (peak_val - baseline) / 2.0
            
            # Find left edge (FWHM)
            left_idx = peak_idx
            for j in range(peak_idx, refined_s - 1, -1):
                if ca_smooth[j] <= half_max:
                    left_idx = j
                    break
            
            # Find right edge (FWHM)
            right_idx = peak_idx
            for j in range(peak_idx, refined_e + 1):
                if ca_smooth[j] <= half_max:
                    right_idx = j
                    break
            
            fwhm_samples = right_idx - left_idx
            fwhm_s = fwhm_samples / float(fs_ca)
            
            transients.append(dict(
                start_idx=int(refined_s),
                end_idx=int(refined_e),
                peak_idx=int(peak_idx),
                baseline=baseline,
                thr=thr,
                peak_amp=float(peak_amp),
                fwhm_samples=int(fwhm_samples),
                fwhm_s=float(fwhm_s)
            ))
    
    return transients, ca_smooth, baseline, thr


def analyze_cell_calcium_transients_hp(
    cell_path,
    trace_vol,
    trace_cal,
    spike_indices,
    frame_rate_vol=500,
    frame_rate_cal=30,
    window_ms=100,
    k=2.0,
    min_dur_s=0.1,
    merge_gap_s=0.15,
    baseline_bins=200,
    median_filter_size_ms=20,
    smooth_sigma_s=0.07,
    name=''
):
    """
    Detect calcium transients and extract aligned voltage traces, spikes, and AUC.
    
    Parameters
    ----------
    cell_path : str
        Path to cell directory (for saving results)
    trace_vol : np.ndarray
        Voltage trace
    trace_cal : np.ndarray
        Calcium trace
    spike_indices : np.ndarray
        Spike times in voltage samples
    frame_rate_vol : int
        Voltage sampling rate (Hz)
    frame_rate_cal : int
        Calcium sampling rate (Hz)
    window_ms : int
        Window around transient peak (±ms)
    k : float
        Threshold multiplier (k * noise_sd above baseline)
    min_dur_s : float
        Minimum transient duration in seconds
    merge_gap_s : float
        Merge transients separated by this gap
    baseline_bins : int
        Number of bins for mode-based baseline estimation
    median_filter_size_ms : float
        Window size in ms for median filtering
    smooth_sigma_s : float
        Gaussian smoothing sigma in seconds for boundary refinement
    name : str
        Suffix for saved files
    
    Returns
    -------
    results_dict : dict with keys:
        - 'transients': list of transient dicts with all metrics
        - 'transient_cal_traces': list of Ca traces around peak
        - 'transient_vol_traces': list of V traces around peak
        - 'transient_spikes': list of spike indices within each transient
        - 'transient_auc': list of AUC values
        - 'transient_auc_above_baseline': list of AUC above baseline
        - 'transient_peak_amplitude': list of peak amplitude values
        - 'transient_fwhm_s': list of FWHM in seconds
        - 'transient_duration_s': list of duration in seconds
        - 'ca_smooth': smoothed Ca trace used for detection
        - 'baseline': baseline value
        - 'threshold': detection threshold
        - 'detection_params': dict of all parameters used
    """
    
    trace_vol = np.asarray(trace_vol, dtype=float)
    trace_cal = np.asarray(trace_cal, dtype=float)
    spike_indices = np.asarray(spike_indices, dtype=int)
    
    # print(f"\n{'='*70}")
    # print(f"CALCIUM TRANSIENT ANALYSIS")
    # print(f"{'='*70}")
    # print(f"Voltage trace: {len(trace_vol)} samples ({len(trace_vol)/frame_rate_vol:.2f} s)")
    # print(f"Calcium trace: {len(trace_cal)} samples ({len(trace_cal)/frame_rate_cal:.2f} s)")
    # print(f"Spikes detected: {len(spike_indices)}")
    
    # ===== 1. DETECT CALCIUM TRANSIENTS =====
    transients, ca_smooth, baseline, thr = find_calcium_transients_hp(
        trace_cal,
        fs_ca=frame_rate_cal,
        k=k,
        min_dur_s=min_dur_s,
        merge_gap_s=merge_gap_s,
        baseline_bins=baseline_bins,
        median_filter_size_ms=median_filter_size_ms,
        smooth_sigma_s=smooth_sigma_s
    )
    
    # print(f"\n✓ Transients detected: {len(transients)}")
    # print(f"  Baseline: {baseline:.4f}")
    # print(f"  Threshold: {thr:.4f}")
    
    # ===== 2. EXTRACT ALIGNED TRACES AND CALCULATE METRICS =====
    transient_cal_traces = []
    transient_vol_traces = []
    transient_spikes_list = []
    transient_auc_list = []
    transient_auc_baseline_list = []
    transient_peak_amp_list = []
    transient_fwhm_list = []
    transient_dur_s_list = []
    
    # Convert window from ms to samples
    window_ca_samples = int(np.round(window_ms * frame_rate_cal / 1000))
    window_vol_samples = int(np.round(window_ms * frame_rate_vol / 1000))
    
    # print(f"\nExtracting ±{window_ms} ms windows around each transient peak:")
    # print(f"  Ca window: ±{window_ca_samples} samples")
    # print(f"  Vol window: ±{window_vol_samples} samples")
    
    # Create time axis for alignment
    t_vol = np.arange(len(trace_vol)) / frame_rate_vol
    t_cal = np.arange(len(trace_cal)) / frame_rate_cal
    
    for trans_idx, transient in enumerate(transients):
        peak_idx_ca = transient['peak_idx']
        start_idx_ca = transient['start_idx']
        end_idx_ca = transient['end_idx']
        
        # Extract Ca window (±window_ms around peak)
        ca_start = max(0, start_idx_ca - window_ca_samples)
        ca_end = min(len(trace_cal), end_idx_ca + window_ca_samples + 1)
        ca_window = trace_cal[ca_start:ca_end]
        
        # Convert peak time to voltage indices
        peak_time_s = t_cal[peak_idx_ca]
        peak_idx_vol = np.argmin(np.abs(t_vol - peak_time_s))
        Start_time_s = t_cal[start_idx_ca]
        Start_idx_vol = np.argmin(np.abs(t_vol - Start_time_s))
        End_time_s = t_cal[end_idx_ca]
        End_idx_vol = np.argmin(np.abs(t_vol - End_time_s))
        
        # Extract Vol window (±window_ms around corresponding time)
        vol_start = max(0, Start_idx_vol - window_vol_samples)
        vol_end = min(len(trace_vol), End_idx_vol + window_vol_samples + 1)
        vol_window = trace_vol[vol_start:vol_end]
        
        # Find spikes within this transient (using original indices)
        spikes_in_event = spike_indices[
            (spike_indices >= Start_idx_vol) & (spike_indices < End_idx_vol)
        ]

        # Keep a version aligned to the extracted voltage window (optional, for plotting)
        # This makes spikes overlay correctly on vol_window (which starts at vol_start)
        spikes_relative_to_window = spikes_in_event - vol_start

        # If you want the saved spike indices to be strictly event-aligned (start=0 at event start):
        spikes_relative_to_event = spikes_in_event - Start_idx_vol
        
        # ===== CALCULATE AUC =====
        # AUC of full calcium transient
        t_ca_transient = np.arange(len(trace_cal[start_idx_ca:end_idx_ca])) / frame_rate_cal
        auc_full = np.trapz(trace_cal[start_idx_ca:end_idx_ca], t_ca_transient)
        
        # AUC above baseline
        t_ca_window = np.arange(len(ca_window)) / frame_rate_cal
        ca_above_baseline = np.maximum(ca_window - baseline, 0)
        auc_above_baseline = np.trapz(ca_above_baseline, t_ca_window)
        
        # Peak amplitude & FWHM from transient dict
        peak_amp = transient['peak_amp']
        fwhm_s = transient['fwhm_s']
        
        # Duration of transient
        trans_dur = (end_idx_ca - start_idx_ca + 1) / frame_rate_cal
        
        # Store results
        transient_cal_traces.append(ca_window)
        transient_vol_traces.append(vol_window)
        transient_spikes_list.append(spikes_relative_to_event)
        transient_auc_list.append(auc_full)
        transient_auc_baseline_list.append(auc_above_baseline)
        transient_peak_amp_list.append(peak_amp)
        transient_fwhm_list.append(fwhm_s)
        transient_dur_s_list.append(trans_dur)
        
        if (trans_idx + 1) % 10 == 0 or trans_idx == len(transients) - 1:
            print(f"  Transient {trans_idx + 1}/{len(transients)}: "
                  f"Peak={peak_amp:.4f} | FWHM={fwhm_s:.3f}s | "
                  f"AUC={auc_full:.2f} | AUC(↑bsl)={auc_above_baseline:.2f} | "
                  f"Duration={trans_dur:.3f}s | Spikes={len(spikes_relative_to_event)}")
    
    # ===== 3. COMPILE RESULTS =====
    results_dict = {
        'transients': transients,
        'transient_cal_traces': transient_cal_traces,
        'transient_vol_traces': transient_vol_traces,
        'transient_spikes': transient_spikes_list,
        'transient_auc': np.array(transient_auc_list, dtype=float),
        'transient_auc_above_baseline': np.array(transient_auc_baseline_list, dtype=float),
        'transient_peak_amplitude': np.array(transient_peak_amp_list, dtype=float),
        'transient_fwhm_s': np.array(transient_fwhm_list, dtype=float),
        'transient_duration_s': np.array(transient_dur_s_list, dtype=float),
        'ca_smooth': ca_smooth,
        'baseline': baseline,
        'threshold': thr,
        'detection_params': {
            'window_ms': window_ms,
            'k': k,
            'min_dur_s': min_dur_s,
            'merge_gap_s': merge_gap_s,
            'baseline_bins': baseline_bins,
            'median_filter_size_ms': median_filter_size_ms,
            'smooth_sigma_s': smooth_sigma_s,
            'frame_rate_vol': frame_rate_vol,
            'frame_rate_cal': frame_rate_cal,
        }
    }
    
    # ===== 4. SAVE RESULTS =====
    save_path = os.path.join(cell_path, f'calcium_transient_analysis{name}.pkl')
    with open(save_path, 'wb') as f:
        pickle.dump(results_dict, f)
    
    # print(f"\n✓ Results saved to: {save_path}")
    # print(f"\nSummary Statistics:")
    # print(f"  Total transients extracted: {len(transient_cal_traces)}")
    # print(f"  Transients with spikes: {sum(1 for s in transient_spikes_list if len(s) > 0)}")
    # print(f"  Total spikes in transients: {sum(len(s) for s in transient_spikes_list)}")
    # print(f"  Mean peak amplitude: {np.nanmean(transient_peak_amp_list):.4f} ± {np.nanstd(transient_peak_amp_list):.4f}")
    # print(f"  Mean FWHM: {np.nanmean(transient_fwhm_list):.3f} ± {np.nanstd(transient_fwhm_list):.3f} s")
    # print(f"  Mean AUC: {np.nanmean(transient_auc_list):.2f} ± {np.nanstd(transient_auc_list):.2f}")
    # print(f"  Mean AUC (above baseline): {np.nanmean(transient_auc_baseline_list):.2f} ± {np.nanstd(transient_auc_baseline_list):.2f}")
    # print(f"  Mean duration: {np.nanmean(transient_dur_s_list):.3f} ± {np.nanstd(transient_dur_s_list):.3f} s")
    
    return results_dict

In [64]:




def add_results_to_state(acc, state, results):
    """
    results: output dict from analyze_cell_calcium_transients_hp
    Adds per-event points:
      amp = peak amplitude
      hwhm = fwhm/2 (seconds)
      nspk = number of spikes (len of spikes_relative list)
    """
    st = state.lower()

    amp = np.asarray(results["transient_peak_amplitude"], float)
    fwhm = np.asarray(results["transient_fwhm_s"], float)
    hwhm = fwhm / 2.0

    # spikes list is list-of-arrays (relative to window start)
    nspk = np.array([len(s) for s in results["transient_spikes"]], float)

    good = np.isfinite(amp) & np.isfinite(hwhm) & np.isfinite(nspk)
    acc[st]["amp"].append(amp[good])
    acc[st]["hwhm"].append(hwhm[good])
    acc[st]["nspk"].append(nspk[good])

def _concat_state(acc, st, key):
    if len(acc[st][key]) == 0:
        return np.array([])
    return np.concatenate(acc[st][key])

def save_matplotlib_svg(fig, out_svg):
    os.makedirs(os.path.dirname(out_svg), exist_ok=True)
    fig.savefig(out_svg, format="svg", bbox_inches="tight")
    print("✅ Saved SVG:", out_svg)


def plotly_state_2x2(acc, x_key, y_key, xlab, ylab, suptitle, state_order=("motor","rest","awake","anst")):
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=[f"{st}" for st in state_order]
    )

    positions = [(1,1),(1,2),(2,1),(2,2)]

    for st, (r,c) in zip(state_order, positions):
        x = np.concatenate(acc[st][x_key]) if len(acc[st][x_key]) else np.array([])
        y = np.concatenate(acc[st][y_key]) if len(acc[st][y_key]) else np.array([])

        if x.size == 0:
            # empty subplot
            fig.add_trace(go.Scatter(x=[], y=[], mode="markers"), row=r, col=c)
            fig.update_xaxes(title_text=xlab, row=r, col=c)
            fig.update_yaxes(title_text=ylab, row=r, col=c)
            continue

        fig.add_trace(
            go.Scatter(
                x=x, y=y,
                mode="markers",
                marker=dict(size=5),  # no color specified
                name=st,
                showlegend=False
            ),
            row=r, col=c
        )

        fig.update_xaxes(title_text=xlab, row=r, col=c)
        fig.update_yaxes(title_text=ylab, row=r, col=c)

        # update subplot title to include n
        fig.layout.annotations[(r-1)*2 + (c-1)].text = f"{st} (n={x.size})"

    fig.update_layout(title_text=suptitle, height=800, width=1000)
    return fig


def save_plotly_html(fig, out_html):
    os.makedirs(os.path.dirname(out_html), exist_ok=True)
    fig.write_html(out_html, include_plotlyjs="cdn")
    print("✅ Saved HTML:", out_html)


In [65]:
DB = pd.read_csv(r'Z:\Adam-Lab-Shared\Data\Michal_Rubin\Dendrites\Pyr.csv')
values = DB['SNR'].tolist()
r = DB
awakePyr = r['Notes']
bsPyr = list(r['brainState'])
pathPyr = list(r['Link'])


In [66]:
# ---- 1) Accumulator for brain states ----
STATE_ORDER = ["motor", "rest", "awake", "anst"]  # 4 panels order

acc = {st: {"amp": [], "hwhm": [], "nspk": []} for st in STATE_ORDER}
for i,l in enumerate(pathPyr):
    print(l)
    TracePathCal = os.path.join(l,'calTraceDF.csv')
    TracePathVol = os.path.join(l,'volTraceDF.csv')
    TracePathCalM = os.path.join(l,'calMask.csv')
    TracePathVolM = os.path.join(l,'volMask.csv')
    parentP = os.path.dirname(l)
    MotPath = os.path.join(parentP,'Sync','MotorId.csv')
    trace_vol_full = np.array(pd.read_csv(TracePathVol)).flatten().astype(float)
    trace_cal_full = np.array(pd.read_csv(TracePathCal)).flatten().astype(float)
    vol_mask = np.array(pd.read_csv(TracePathVolM)).flatten().astype(bool)
    cal_mask = np.array(pd.read_csv(TracePathCalM)).flatten().astype(bool)
    # Apply masks
    trace_vol = trace_vol_full[vol_mask]
    trace_cal = trace_cal_full[cal_mask]
    motor = pd.read_csv(MotPath, header=None).iloc[:, 0]
    motor = motor[0:np.size(trace_vol,0)]
    VolAX = np.linspace(0, (len(trace_vol)/500), len(trace_vol)) 
    CalAX = np.linspace(0, (len(trace_cal)/30), len(trace_cal))
    
   
    if bsPyr[i].lower()=='motor':
        TracePathCal = os.path.join(l,'changepoint.csv')
        changeP = pd.read_csv(TracePathCal)
        changeP = np.array(changeP)
        changeP = np.array(changeP).flatten().tolist()
        #calMot,calRes, spikeMot,spikeRes,vmot,volR = Split_cal(changeP,trace_vol,trace_cal,VolAX,CalAX,motor,spike_indices)
        
        RestFRpath =os.path.join(l,r'FiringRateRest.csv')
        MotorFRpath =os.path.join(l,r'FiringRateMotor.csv')
        dfMot = pd.read_csv(MotorFRpath)
        MotorFR = dfMot['fr'].tolist()
        dfRest = pd.read_csv(RestFRpath)
        RestFR = dfRest['fr'].tolist()
        for k in range(len(MotorFR)):
            TracePathVol = os.path.join(l,f'volTraceMot{k}.csv')
            TracePathCal = os.path.join(l,f'calTraceMot{k}.csv')
            trace_vol_full = np.array(pd.read_csv(TracePathVol)).flatten().astype(float)
            trace_cal_full = np.array(pd.read_csv(TracePathCal)).flatten().astype(float)
            VolAX = np.linspace(0, (len(trace_vol_full)/500), len(trace_vol_full)) 
            CalAX = np.linspace(0, (len(trace_cal_full)/30), len(trace_cal_full))
            N = f'm{k}'
            if len(trace_vol_full) < 13 * 500:
                #print(f"SKIP {l} r{k}: VolTrace too short ({len(trace_vol_full)/500:.2f} s, {len(trace_vol_full)} frames)")
                continue
            curr_path_pkl = os.path.join(l, f'spike_detection_refined{N}.pkl')
            if os.path.exists(curr_path_pkl):
                with open(curr_path_pkl, 'rb') as f:
                    data = pickle.load(f)
                spike_indices = data['vm_all_spikes']
                
                params_list = data['detection_params']

                # if len(params_list) > 0:
                #     prev_params = params_list[-1]   # last used params
                #     init_CS = prev_params['pnorm_CS']
                #     init_SS = prev_params['pnorm_SS']
                #     init_thresh = prev_params['threshold']

                # # Sort to ensure chronological order
                # currSp = sorted(list(set(currSp)))
                # currSp = sorted(currSp)
            # else:
            #     currSp = spikeMot[k].tolist()
            
            plt.close('all')
            results = analyze_cell_calcium_transients(
                cell_path=l,
                trace_vol=trace_vol_full,
                trace_cal=trace_cal_full,
                spike_indices=spike_indices,
                frame_rate_vol=500,
                frame_rate_cal=30,
                window_ms=100,
                name=N
            )
            fig = plot_ca_vol_with_transients_and_spikes(
                ca=results['ca_smooth'],
                vol=trace_vol_full,
                fs_ca=30,
                fs_vol=500,
                spike_idx_v=spike_indices,
                transients=results['transients'],
                title="Full Trace: Calcium + Voltage with Spikes and Transients"
            )

            # Save as HTML
            fig.write_html(os.path.join(l, f'full_trace_with_transients_{N}.html'))
            add_results_to_state(acc, state="rest", results=results)

        for j in range(len(RestFR)):
            TracePathVol = os.path.join(l,f'volTraceRest{j}.csv')
            TracePathCal = os.path.join(l,f'calTraceRest{j}.csv')
            trace_vol_full = np.array(pd.read_csv(TracePathVol)).flatten().astype(float)
            trace_cal_full = np.array(pd.read_csv(TracePathCal)).flatten().astype(float)
            VolAX = np.linspace(0, (len(trace_vol_full)/500), len(trace_vol_full)) 
            CalAX = np.linspace(0, (len(trace_cal_full)/30), len(trace_cal_full))
            if len(trace_vol_full) < 13 * 500:
                #print(f"SKIP {l} r{k}: VolTrace too short ({len(trace_vol_full)/500:.2f} s, {len(trace_vol_full)} frames)")
                continue
            N = f'r{j}'
            curr_path_pkl = os.path.join(l, f'spike_detection_refined{N}.pkl')
            if os.path.exists(curr_path_pkl):
                with open(curr_path_pkl, 'rb') as f:
                    data = pickle.load(f)
                spike_indices = data['vm_all_spikes']
                
                params_list = data['detection_params']

                # if len(params_list) > 0:
                #     prev_params = params_list[-1]   # last used params
                #     init_CS = prev_params['pnorm_CS']
                #     init_SS = prev_params['pnorm_SS']
                #     init_thresh = prev_params['threshold']

                # # Sort to ensure chronological order
                # currSp = sorted(list(set(currSp)))
                # currSp = sorted(currSp)
            # else:
            #     currSp = spikeMot[k].tolist()
            
            plt.close('all')
            results = analyze_cell_calcium_transients(
                cell_path=l,
                trace_vol=trace_vol_full,
                trace_cal=trace_cal_full,
                spike_indices=spike_indices,
                frame_rate_vol=500,
                frame_rate_cal=30,
                window_ms=100,
                name=N
            )
            fig = plot_ca_vol_with_transients_and_spikes(
                ca=results['ca_smooth'],
                vol=trace_vol_full,
                fs_ca=30,
                fs_vol=500,
                spike_idx_v=spike_indices,
                transients=results['transients'],
                title="Full Trace: Calcium + Voltage with Spikes and Transients"
            )

            # Save as HTML
            fig.write_html(os.path.join(l, f'full_trace_with_transients_{N}.html'))
            add_results_to_state(acc, state="rest", results=results)

    elif bsPyr[i].lower()=='awake':
        spikePath = os.path.join(l, 'spike_detection_refined.pkl')
        if os.path.exists(spikePath):
            with open(spikePath, 'rb') as f:
                spike_data = pickle.load(f)
            spike_indices = spike_data['vm_all_spikes']
        else:
            path =os.path.join(l,r'cell__CS_detection.pkl')
            with open(path, 'rb') as f:
                data = pickle.load(f)

            raw_spikes_structure = data['all_spikes']
            spike_indices = []
            
            if isinstance(raw_spikes_structure, list):
                for sublist in raw_spikes_structure:
                    if isinstance(sublist, (list, np.ndarray)):
                        spike_indices.extend(sublist)
                    else:
                        spike_indices.append(sublist)
            else:
                spike_indices = list(raw_spikes_structure)
        plt.close('all')
        results = analyze_cell_calcium_transients(
                cell_path=l,
                trace_vol=trace_vol,
                trace_cal=trace_cal,
                spike_indices=spike_indices,
                frame_rate_vol=500,
                frame_rate_cal=30,
                window_ms=100
            )
        fig = plot_ca_vol_with_transients_and_spikes(
            ca=results['ca_smooth'],
            vol=trace_vol_full,
            fs_ca=30,
            fs_vol=500,
            spike_idx_v=spike_indices,
            transients=results['transients'],
            title="Full Trace: Calcium + Voltage with Spikes and Transients"
        )

        # Save as HTML
        fig.write_html(os.path.join(l, f'full_trace_with_transients.html'))
        add_results_to_state(acc, state="awake", results=results)

    elif bsPyr[i].lower()=='anst':
        spikePath = os.path.join(l, 'spike_detection_refined.pkl')
        spikePath2 = os.path.join(l, 'spike_detection_refinedr1.pkl')
        if os.path.exists(spikePath) or os.path.exists(spikePath2):
            if os.path.exists(spikePath):
                with open(spikePath, 'rb') as f:
                    spike_data = pickle.load(f)
                spike_indices = spike_data['vm_all_spikes']
            else:
                with open(spikePath2, 'rb') as f:
                    spike_data = pickle.load(f)
                spike_indices = spike_data['vm_all_spikes']
        else:
            path =os.path.join(l,r'cell__CS_detection.pkl')
            with open(path, 'rb') as f:
                data = pickle.load(f)                       
            
            raw_spikes_structure = data['all_spikes']
            spike_indices = []
            
            if isinstance(raw_spikes_structure, list):
                for sublist in raw_spikes_structure:
                    if isinstance(sublist, (list, np.ndarray)):
                        spike_indices.extend(sublist)
                    else:
                        spike_indices.append(sublist)
            else:
                spike_indices = list(raw_spikes_structure)
        results = analyze_cell_calcium_transients(
                cell_path=l,
                trace_vol=trace_vol,
                trace_cal=trace_cal,
                spike_indices=spike_indices,
                frame_rate_vol=500,
                frame_rate_cal=30,
                window_ms=100
            )
        fig = plot_ca_vol_with_transients_and_spikes(
            ca=results['ca_smooth'],
            vol=trace_vol_full,
            fs_ca=30,
            fs_vol=500,
            spike_idx_v=spike_indices,
            transients=results['transients'],
            title="Full Trace: Calcium + Voltage with Spikes and Transients"
        )

        # Save as HTML
        fig.write_html(os.path.join(l, f'full_trace_with_transients.html'))
        add_results_to_state(acc, state="anst", results=results)


Z:\Adam-Lab-Shared\Data\Michal_Rubin\rugc42\Wh\21-10-2025-MOTOR\fov7\cell0
  Transient 6/6: AUC=0.56 | AUC(↑baseline)=0.46 | Peak=1.1580 | Duration=0.633s | Spikes=12
  Transient 5/5: AUC=0.32 | AUC(↑baseline)=0.33 | Peak=1.4633 | Duration=0.433s | Spikes=6
Z:\Adam-Lab-Shared\Data\Michal_Rubin\RUGC40\R\18-08-2025-ans\fov1\cell0
  Transient 10/29: AUC=0.56 | AUC(↑baseline)=0.54 | Peak=2.2780 | Duration=0.500s | Spikes=4
  Transient 20/29: AUC=0.43 | AUC(↑baseline)=0.41 | Peak=1.9454 | Duration=0.467s | Spikes=3
  Transient 29/29: AUC=0.40 | AUC(↑baseline)=0.41 | Peak=1.8147 | Duration=0.400s | Spikes=4
Z:\Adam-Lab-Shared\Data\Michal_Rubin\RUGC40\R\18-08-2025-ans\fov2\cell1
  Transient 10/29: AUC=0.51 | AUC(↑baseline)=0.52 | Peak=2.0592 | Duration=0.400s | Spikes=7
  Transient 20/29: AUC=0.51 | AUC(↑baseline)=0.58 | Peak=2.1925 | Duration=0.333s | Spikes=4
  Transient 29/29: AUC=0.55 | AUC(↑baseline)=0.58 | Peak=2.2080 | Duration=0.367s | Spikes=4
Z:\Adam-Lab-Shared\Data\Michal_Rubin\RUG

In [67]:
# ---- 1) Accumulator for brain states ----
STATE_ORDER_hp = ["motor", "rest", "awake", "anst"]  # 4 panels order

acc_hp = {st: {"amp": [], "hwhm": [], "nspk": []} for st in STATE_ORDER}
for i,l in enumerate(pathPyr):
    print(l)
    TracePathCal = os.path.join(l,'calTraceDF.csv')
    TracePathVol = os.path.join(l,'volTraceDF.csv')
    TracePathCalM = os.path.join(l,'calMask.csv')
    TracePathVolM = os.path.join(l,'volMask.csv')
    parentP = os.path.dirname(l)
    MotPath = os.path.join(parentP,'Sync','MotorId.csv')
    trace_vol_full = np.array(pd.read_csv(TracePathVol)).flatten().astype(float)
    trace_cal_full = np.array(pd.read_csv(TracePathCal)).flatten().astype(float)
    vol_mask = np.array(pd.read_csv(TracePathVolM)).flatten().astype(bool)
    cal_mask = np.array(pd.read_csv(TracePathCalM)).flatten().astype(bool)
    # Apply masks
    trace_vol = trace_vol_full[vol_mask]
    trace_cal = trace_cal_full[cal_mask]
    motor = pd.read_csv(MotPath, header=None).iloc[:, 0]
    motor = motor[0:np.size(trace_vol,0)]
    VolAX = np.linspace(0, (len(trace_vol)/500), len(trace_vol)) 
    CalAX = np.linspace(0, (len(trace_cal)/30), len(trace_cal))
    
   
    if bsPyr[i].lower()=='motor':
        TracePathCal = os.path.join(l,'changepoint.csv')
        changeP = pd.read_csv(TracePathCal)
        changeP = np.array(changeP)
        changeP = np.array(changeP).flatten().tolist()
        #calMot,calRes, spikeMot,spikeRes,vmot,volR = Split_cal(changeP,trace_vol,trace_cal,VolAX,CalAX,motor,spike_indices)
        
        RestFRpath =os.path.join(l,r'FiringRateRest.csv')
        MotorFRpath =os.path.join(l,r'FiringRateMotor.csv')
        dfMot = pd.read_csv(MotorFRpath)
        MotorFR = dfMot['fr'].tolist()
        dfRest = pd.read_csv(RestFRpath)
        RestFR = dfRest['fr'].tolist()
        for k in range(len(MotorFR)):
            TracePathVol = os.path.join(l,f'volTraceMot{k}.csv')
            TracePathCal = os.path.join(l,f'calTraceMot{k}.csv')
            trace_vol_full = np.array(pd.read_csv(TracePathVol)).flatten().astype(float)
            trace_cal_full = np.array(pd.read_csv(TracePathCal)).flatten().astype(float)
            VolAX = np.linspace(0, (len(trace_vol_full)/500), len(trace_vol_full)) 
            CalAX = np.linspace(0, (len(trace_cal_full)/30), len(trace_cal_full))
            N = f'm{k}'
            if len(trace_vol_full) < 13 * 500:
                #print(f"SKIP {l} r{k}: VolTrace too short ({len(trace_vol_full)/500:.2f} s, {len(trace_vol_full)} frames)")
                continue
            curr_path_pkl = os.path.join(l, f'spike_detection_refined{N}.pkl')
            if os.path.exists(curr_path_pkl):
                with open(curr_path_pkl, 'rb') as f:
                    data = pickle.load(f)
                spike_indices = data['vm_all_spikes']
                
                params_list = data['detection_params']

                # if len(params_list) > 0:
                #     prev_params = params_list[-1]   # last used params
                #     init_CS = prev_params['pnorm_CS']
                #     init_SS = prev_params['pnorm_SS']
                #     init_thresh = prev_params['threshold']

                # # Sort to ensure chronological order
                # currSp = sorted(list(set(currSp)))
                # currSp = sorted(currSp)
            # else:
            #     currSp = spikeMot[k].tolist()
            
            plt.close('all')
            results = analyze_cell_calcium_transients_hp(
                cell_path=l,
                trace_vol=trace_vol_full,
                trace_cal=trace_cal_full,
                spike_indices=spike_indices,
                frame_rate_vol=500,
                frame_rate_cal=30,
                window_ms=100,
                name=N
            )
            fig = plot_ca_vol_with_transients_and_spikes(
                ca=results['ca_smooth'],
                vol=trace_vol_full,
                fs_ca=30,
                fs_vol=500,
                spike_idx_v=spike_indices,
                transients=results['transients'],
                title="Full Trace: Calcium + Voltage with Spikes and Transients"
            )

            # Save as HTML
            fig.write_html(os.path.join(l, f'full_trace_with_transients_{N}_hp_det.html'))
            add_results_to_state(acc_hp, state="motor", results=results)
        for j in range(len(RestFR)):
            TracePathVol = os.path.join(l,f'volTraceRest{j}.csv')
            TracePathCal = os.path.join(l,f'calTraceRest{j}.csv')
            trace_vol_full = np.array(pd.read_csv(TracePathVol)).flatten().astype(float)
            trace_cal_full = np.array(pd.read_csv(TracePathCal)).flatten().astype(float)
            VolAX = np.linspace(0, (len(trace_vol_full)/500), len(trace_vol_full)) 
            CalAX = np.linspace(0, (len(trace_cal_full)/30), len(trace_cal_full))
            if len(trace_vol_full) < 13 * 500:
                #print(f"SKIP {l} r{k}: VolTrace too short ({len(trace_vol_full)/500:.2f} s, {len(trace_vol_full)} frames)")
                continue
            N = f'r{j}'
            curr_path_pkl = os.path.join(l, f'spike_detection_refined{N}.pkl')
            if os.path.exists(curr_path_pkl):
                with open(curr_path_pkl, 'rb') as f:
                    data = pickle.load(f)
                spike_indices = data['vm_all_spikes']
                
                params_list = data['detection_params']

                # if len(params_list) > 0:
                #     prev_params = params_list[-1]   # last used params
                #     init_CS = prev_params['pnorm_CS']
                #     init_SS = prev_params['pnorm_SS']
                #     init_thresh = prev_params['threshold']

                # # Sort to ensure chronological order
                # currSp = sorted(list(set(currSp)))
                # currSp = sorted(currSp)
            # else:
            #     currSp = spikeMot[k].tolist()
            
            plt.close('all')
            results = analyze_cell_calcium_transients_hp(
                cell_path=l,
                trace_vol=trace_vol_full,
                trace_cal=trace_cal_full,
                spike_indices=spike_indices,
                frame_rate_vol=500,
                frame_rate_cal=30,
                window_ms=100,
                name=N
            )
            fig = plot_ca_vol_with_transients_and_spikes(
                ca=results['ca_smooth'],
                vol=trace_vol_full,
                fs_ca=30,
                fs_vol=500,
                spike_idx_v=spike_indices,
                transients=results['transients'],
                title="Full Trace: Calcium + Voltage with Spikes and Transients"
            )

            # Save as HTML
            fig.write_html(os.path.join(l, f'full_trace_with_transients_{N}_hp_det.html'))
            add_results_to_state(acc_hp, state="rest", results=results)
    elif bsPyr[i].lower()=='awake':
        spikePath = os.path.join(l, 'spike_detection_refined.pkl')
        if os.path.exists(spikePath):
            with open(spikePath, 'rb') as f:
                spike_data = pickle.load(f)
            spike_indices = spike_data['vm_all_spikes']
        else:
            path =os.path.join(l,r'cell__CS_detection.pkl')
            with open(path, 'rb') as f:
                data = pickle.load(f)

            raw_spikes_structure = data['all_spikes']
            spike_indices = []
            
            if isinstance(raw_spikes_structure, list):
                for sublist in raw_spikes_structure:
                    if isinstance(sublist, (list, np.ndarray)):
                        spike_indices.extend(sublist)
                    else:
                        spike_indices.append(sublist)
            else:
                spike_indices = list(raw_spikes_structure)
        plt.close('all')
        results = analyze_cell_calcium_transients_hp(
                cell_path=l,
                trace_vol=trace_vol,
                trace_cal=trace_cal,
                spike_indices=spike_indices,
                frame_rate_vol=500,
                frame_rate_cal=30,
                window_ms=100
            )
        fig = plot_ca_vol_with_transients_and_spikes(
            ca=results['ca_smooth'],
            vol=trace_vol_full,
            fs_ca=30,
            fs_vol=500,
            spike_idx_v=spike_indices,
            transients=results['transients'],
            title="Full Trace: Calcium + Voltage with Spikes and Transients"
        )

        # Save as HTML
        fig.write_html(os.path.join(l, f'full_trace_with_transients_hp_det.html'))
        add_results_to_state(acc_hp, state="awake", results=results)    
    elif bsPyr[i].lower()=='anst':
        spikePath = os.path.join(l, 'spike_detection_refined.pkl')
        spikePath2 = os.path.join(l, 'spike_detection_refinedr1.pkl')
        if os.path.exists(spikePath) or os.path.exists(spikePath2):
            if os.path.exists(spikePath):
                with open(spikePath, 'rb') as f:
                    spike_data = pickle.load(f)
                spike_indices = spike_data['vm_all_spikes']
            else:
                with open(spikePath2, 'rb') as f:
                    spike_data = pickle.load(f)
                spike_indices = spike_data['vm_all_spikes']
        else:
            path =os.path.join(l,r'cell__CS_detection.pkl')
            with open(path, 'rb') as f:
                data = pickle.load(f)                       
            
            raw_spikes_structure = data['all_spikes']
            spike_indices = []
            
            if isinstance(raw_spikes_structure, list):
                for sublist in raw_spikes_structure:
                    if isinstance(sublist, (list, np.ndarray)):
                        spike_indices.extend(sublist)
                    else:
                        spike_indices.append(sublist)
            else:
                spike_indices = list(raw_spikes_structure)
        results = analyze_cell_calcium_transients_hp(
                cell_path=l,
                trace_vol=trace_vol,
                trace_cal=trace_cal,
                spike_indices=spike_indices,
                frame_rate_vol=500,
                frame_rate_cal=30,
                window_ms=100
            )
        fig = plot_ca_vol_with_transients_and_spikes(
            ca=results['ca_smooth'],
            vol=trace_vol_full,
            fs_ca=30,
            fs_vol=500,
            spike_idx_v=spike_indices,
            transients=results['transients'],
            title="Full Trace: Calcium + Voltage with Spikes and Transients"
        )

        # Save as HTML
        fig.write_html(os.path.join(l, f'full_trace_with_transients_hp_det.html'))
        add_results_to_state(acc_hp, state="anst", results=results)

Z:\Adam-Lab-Shared\Data\Michal_Rubin\rugc42\Wh\21-10-2025-MOTOR\fov7\cell0
  Transient 10/14: Peak=0.5954 | FWHM=0.000s | AUC=0.75 | AUC(↑bsl)=0.52 | Duration=0.900s | Spikes=12
  Transient 14/14: Peak=0.4828 | FWHM=0.000s | AUC=0.24 | AUC(↑bsl)=0.17 | Duration=0.333s | Spikes=2
  Transient 10/11: Peak=0.4091 | FWHM=0.000s | AUC=0.37 | AUC(↑bsl)=0.29 | Duration=0.967s | Spikes=15
  Transient 11/11: Peak=0.3256 | FWHM=0.000s | AUC=0.37 | AUC(↑bsl)=0.29 | Duration=0.967s | Spikes=15
Z:\Adam-Lab-Shared\Data\Michal_Rubin\RUGC40\R\18-08-2025-ans\fov1\cell0
  Transient 10/39: Peak=1.6242 | FWHM=0.533s | AUC=1.06 | AUC(↑bsl)=0.83 | Duration=1.100s | Spikes=5
  Transient 20/39: Peak=0.8462 | FWHM=0.000s | AUC=0.34 | AUC(↑bsl)=0.25 | Duration=0.467s | Spikes=4
  Transient 30/39: Peak=0.6232 | FWHM=0.000s | AUC=1.14 | AUC(↑bsl)=0.85 | Duration=1.333s | Spikes=8
  Transient 39/39: Peak=1.0263 | FWHM=0.167s | AUC=0.29 | AUC(↑bsl)=0.25 | Duration=0.333s | Spikes=4
Z:\Adam-Lab-Shared\Data\Michal_Rub

In [68]:
# ============================================================
# PLOTTING AFTER PROCESSING ALL CELLS
# ============================================================

out_dir = r"Z:\Adam-Lab-Shared\Data\Michal_Rubin\data summery\2026\Pyr\calcium_based"
os.makedirs(out_dir, exist_ok=True)

# ------------------------------------------------------------
# FIGURE 1: Amplitude vs HWHM
# ------------------------------------------------------------
fig_amp_hwhm = plotly_state_2x2(
    acc,
    x_key="hwhm",
    y_key="amp",
    xlab="HWHM (s)  [= FWHM / 2]",
    ylab="Calcium peak amplitude",
    suptitle="Amplitude vs HWHM (per brain state)"
)

fig_amp_hwhm.update_layout(width=1000, height=800)

# Save SVG (Plotly → SVG, requires kaleido)
fig_amp_hwhm.write_image(
    os.path.join(out_dir, "Amp_vs_HWHM_states.svg"),
    format="svg"
)

# Save HTML
fig_amp_hwhm.write_html(
    os.path.join(out_dir, "Amp_vs_HWHM_states.html"),
    include_plotlyjs="cdn"
)



# ------------------------------------------------------------
# FIGURE 2: Amplitude vs spike count (STRICTLY under Ca event)
# ------------------------------------------------------------
fig_amp_spk = plotly_state_2x2(
    acc,
    x_key="nspk",
    y_key="amp",
    xlab="# spikes during Ca event",
    ylab="Calcium peak amplitude",
    suptitle="Amplitude vs spike count (per brain state)"
)

# Save SVG (Plotly → SVG)
fig_amp_spk.write_image(
    os.path.join(out_dir, "Amp_vs_SpikeCount_states.svg"),
    format="svg"
)

# Save HTML
fig_amp_spk.write_html(
    os.path.join(out_dir, "Amp_vs_SpikeCount_states.html"),
    include_plotlyjs="cdn"
)


# HWHM vs spike count (per state)
fig_hwhm_spk = plotly_state_2x2(
    acc,
    x_key="nspk",
    y_key="hwhm",
    xlab="# spikes during Ca event",
    ylab="HWHM (s)  [= FWHM/2]",
    suptitle="HWHM vs spike count (per brain state)"
)

fig_hwhm_spk.update_layout(width=1000, height=800)

# Save HTML
fig_hwhm_spk.write_html(
    os.path.join(out_dir, "HWHM_vs_SpikeCount_states.html"),
    include_plotlyjs="cdn"
)

# Save SVG (requires kaleido)
fig_hwhm_spk.write_image(
    os.path.join(out_dir, "HWHM_vs_SpikeCount_states.svg"),
    format="svg"
)

c:\Users\owner\AppData\Local\Programs\Python\Python310\lib\site-packages\kaleido\scopes\base.py:185: DeprecationWarning:

setDaemon() is deprecated, set the daemon attribute instead



In [69]:
# ============================================================
# PLOTTING AFTER PROCESSING ALL CELLS
# ============================================================

out_dir = r"Z:\Adam-Lab-Shared\Data\Michal_Rubin\data summery\2026\Pyr\calcium_based"
os.makedirs(out_dir, exist_ok=True)

# ------------------------------------------------------------
# FIGURE 1: Amplitude vs HWHM
# ------------------------------------------------------------
fig_amp_hwhm = plotly_state_2x2(
    acc=acc_hp,
    x_key="hwhm",
    y_key="amp",
    xlab="HWHM (s)  [= FWHM / 2]",
    ylab="Calcium peak amplitude",
    suptitle="Amplitude vs HWHM (per brain state)"
)

fig_amp_hwhm.update_layout(width=1000, height=800)

# Save SVG (Plotly → SVG, requires kaleido)
fig_amp_hwhm.write_image(
    os.path.join(out_dir, "Amp_vs_HWHM_states_hp.svg"),
    format="svg"
)

# Save HTML
fig_amp_hwhm.write_html(
    os.path.join(out_dir, "Amp_vs_HWHM_states_hp.html"),
    include_plotlyjs="cdn"
)



# ------------------------------------------------------------
# FIGURE 2: Amplitude vs spike count (STRICTLY under Ca event)
# ------------------------------------------------------------
fig_amp_spk = plotly_state_2x2(
    acc = acc_hp,
    x_key="nspk",
    y_key="amp",
    xlab="# spikes during Ca event",
    ylab="Calcium peak amplitude",
    suptitle="Amplitude vs spike count (per brain state)"
)

# Save SVG (Plotly → SVG)
fig_amp_spk.write_image(
    os.path.join(out_dir, "Amp_vs_SpikeCount_states_hp.svg"),
    format="svg"
)

# Save HTML
fig_amp_spk.write_html(
    os.path.join(out_dir, "Amp_vs_SpikeCount_states_hp.html"),
    include_plotlyjs="cdn"
)


# HWHM vs spike count (per state)
fig_hwhm_spk = plotly_state_2x2(
    acc = acc_hp,
    x_key="nspk",
    y_key="hwhm",
    xlab="# spikes during Ca event",
    ylab="HWHM (s)  [= FWHM/2]",
    suptitle="HWHM vs spike count (per brain state)"
)

fig_hwhm_spk.update_layout(width=1000, height=800)

# Save HTML
fig_hwhm_spk.write_html(
    os.path.join(out_dir, "HWHM_vs_SpikeCount_states_hp.html"),
    include_plotlyjs="cdn"
)

# Save SVG (requires kaleido)
fig_hwhm_spk.write_image(
    os.path.join(out_dir, "HWHM_vs_SpikeCount_states_hp.svg"),
    format="svg"
)